# 17 — Silver-label smoke test (1 topic, Batch API)

End-to-end test of the Claude silver-labeling pipeline from `PROPOSAL_COASTAL_TOPIC_ANALYSIS_V2.md`
on a **tiny sample (default: 1 topic)** before firing the real batches
(**phase 1: 6 coast runs, 209 topics — phase 2: all 20 runs, 525 topics**).

Only per-language runs exist (`coast_band_A_en`, `coast_band_A_vi`, …) — mixed-language
run_ids were removed from the database and are not part of the analysis.

Everything goes through the **Message Batches API** (50% off) — same code path as the full run.
A 1-request batch typically finishes in a couple of minutes.

Prerequisite: `CLAUDE_API_KEY=sk-ant-...` in the repo-root `.env`.

In [ ]:
import sys, json, time
sys.path.append("../src")

import duckdb
import llm_label as ll

# ---- knobs -------------------------------------------------------------
RUN_ID = "coast_band_A_en"   # any checkpointed run_id
N_TOPICS = 1                  # how many topics to test-label
# -------------------------------------------------------------------------

con = duckdb.connect(str(ll.DB_PATH))
print("DB:", ll.DB_PATH)
print("Current (checkpointed) run_ids:", ll.current_run_ids(con))

## 1. Prepare schema (idempotent)
Adds the silver-label columns to `TOPIC_LABELS` and creates `TOPIC_ASPECTS`.

In [ ]:
ll.ensure_label_tables(con)
con.execute("DESCRIBE TOPIC_ASPECTS").df()

## 2. Pick the sample topic(s)
Take the biggest topic(s) of the run — most documents, so the label is easy to eyeball.

In [ ]:
sample = con.execute("""
    SELECT topic_id, top_words, n_docs
    FROM TOPIC_LABELS
    WHERE run_id = ? AND topic_id != -1
    ORDER BY n_docs DESC NULLS LAST, topic_id
    LIMIT ?
""", [RUN_ID, N_TOPICS]).fetchall()

for tid, words, n in sample:
    print(f"topic {tid}  ({n} docs)")
    print("  top words:", ", ".join((words or "").split(",")[:10]))
    for d in ll.sample_docs(con, RUN_ID, tid):
        print("  doc:", d[:120].replace("\n", " "), "...")
    print()

## 3. Build the batch request(s) and preview the prompt

In [ ]:
requests = [
    ll.build_request(RUN_ID, tid, words or "", ll.sample_docs(con, RUN_ID, tid))
    for tid, words, _ in sample
]

print("custom_id:", requests[0]["custom_id"])
print("model    :", requests[0]["params"]["model"])
print("\n--- user prompt of first request ---\n")
print(requests[0]["params"]["messages"][0]["content"])

### Token / cost preview (free — `count_tokens` costs nothing)

In [ ]:
client = ll.get_client()

ct = client.messages.count_tokens(
    model=ll.MODEL,
    system=ll.SYSTEM_PROMPT,
    messages=requests[0]["params"]["messages"],
)
in_tok = ct.input_tokens
est_out = 350
per_topic = in_tok / 1e6 * 1.50 + est_out / 1e6 * 7.50

n_coast = con.execute(
    "SELECT COUNT(*) FROM TOPIC_LABELS WHERE run_id LIKE 'coast_band_%' AND topic_id != -1"
).fetchone()[0]
n_all = con.execute(
    "SELECT COUNT(*) FROM TOPIC_LABELS WHERE topic_id != -1"
).fetchone()[0]

print(f"input tokens / request   : {in_tok}")
print(f"batch cost / request     : ${per_topic:.5f}  (assuming ~{est_out} output tokens)")
print(f"coast runs ({n_coast} topics)   : ${n_coast * per_topic:.2f}")
print(f"all runs   ({n_all} topics)   : ${n_all * per_topic:.2f}")

## 4. Submit the mini-batch

In [ ]:
batch = client.messages.batches.create(requests=requests)
print("batch id:", batch.id)
print("status  :", batch.processing_status)

## 5. Poll until the batch ends
Small batches usually finish in 1–5 minutes.

In [ ]:
while True:
    batch = client.messages.batches.retrieve(batch.id)
    c = batch.request_counts
    print(f"{batch.processing_status}  processing={c.processing} succeeded={c.succeeded} errored={c.errored}")
    if batch.processing_status == "ended":
        break
    time.sleep(15)

## 6. Parse the result(s)

In [ ]:
labels = {}
for result in client.messages.batches.results(batch.id):
    if result.result.type != "succeeded":
        print("FAILED:", result.custom_id, result.result.type)
        continue
    msg = result.result.message
    text = next(b.text for b in msg.content if b.type == "text")
    label = ll.parse_label(text)
    labels[result.custom_id] = label
    print(f"=== {result.custom_id} ===")
    print(json.dumps(label, indent=2, ensure_ascii=False))
    print(f"(usage: {msg.usage.input_tokens} in / {msg.usage.output_tokens} out)")

**Sanity checklist before scaling up:**
- `key_aspect` values make sense for the top words / excerpts
- per-aspect `sentiment` matches the excerpt tone
- `sub_aspects` are 1–2 word snake_case
- weights sum to 1.0, single-aspect topics aren't padded with extra aspects

## 7. Write to DuckDB and read it back

In [ ]:
for custom_id, label in labels.items():
    run_id, topic_id = ll.parse_custom_id(custom_id)
    ll.write_label(con, run_id, topic_id, label)
    print("wrote", custom_id)

con.execute("""
    SELECT ta.run_id, ta.topic_id, ta.key_aspect, ta.is_primary,
           ROUND(ta.weight, 2) AS weight, ta.sentiment, ta.sub_aspects,
           tl.confidence, tl.short_reason
    FROM TOPIC_ASPECTS ta
    JOIN TOPIC_LABELS tl ON ta.run_id = tl.run_id AND ta.topic_id = tl.topic_id
    WHERE ta.run_id = ?
    ORDER BY ta.topic_id, ta.is_primary DESC
""", [RUN_ID]).df()

## 8. Scale up

When the label above looks good, run the real job from the terminal.

**Phase 1 — coast bands first (209 topics, ≈ $1.05):**

```bash
uv run python src/llm_label_submit.py coast_band_A_en coast_band_A_vi coast_band_B_en coast_band_B_vi coast_band_C_en coast_band_C_vi
```

**Phase 2 — everything (all 20 runs, 525 topics, ≈ $2.70; already-labeled coast topics are skipped):**

```bash
uv run python src/llm_label_submit.py --all
```

**Collect results** (later, even after a reboot):

```bash
uv run python src/llm_label_retrieve.py            # or add --wait to block
```

The submit script rejects mixed run_ids (`coast_band_A`, `year_2020`) — only `*_en` / `*_vi` are valid.
Already-labeled topics (like the one tested here) are skipped automatically — use `--force` to re-label.

In [ ]:
con.close()